<a href="https://colab.research.google.com/github/chanita-lab/Code-repository/blob/main/Lab_1__Pan_sharpening_6606614680_%E0%B8%8A%E0%B8%99%E0%B8%B4%E0%B8%95%E0%B8%B2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **Pan-sharpening**

เทคนิค Pan-sharpening เป็นเทคนิคที่ใช้กับภาพถ่ายดาวเทียม ที่มี Panchromatic band เช่น THEOS-1, LANDSAT-8, LANDSAT-9 เพื่อให้มีรายละเอียดและมีประโยชน์มากขึ้น โดย Lab นี้เราจะทดลองกับข้อมูลดาวเทียม Landsat-9

ข้อมูลถ่ายที่เป็น Multi-Spectral มักจะมีรายละเอียดภาพไม่ค่อยดีนัก เมื่อเทียบกับข้อมูล Panchromatics  ดังนั้น การปรับความคมชัดของภาพช่วยได้โดยการรวมภาพเหล่านี้เข้ากับภาพขาวดำพิเศษที่มีรายละเอียดมากขึ้น ภาพขาวดำที่เรียกว่าภาพแพนโครมาติก มีรายละเอียดในระดับที่สูงกว่า แต่ไม่มีสี เราต้องการใช้รายละเอียดจากภาพแพนโครมาติกและสีจากภาพ Landsat-9 เพื่อสร้างภาพสีที่มีรายละเอียดสูงใหม่

ในการดำเนินการใน Lab นี้ นักศึกษา จำเป็นต้องตรวจสอบให้แน่ใจว่าข้อมูลทั้ง Panchromatic และ Multispectral ของ Landsat-9 อยู่ในตำแหน่งที่สมบูรณ์แบบ เพื่อให้เข้ากันได้อย่างลงตัว

มีหลายวิธีในการรวมภาพเหล่านี้ เช่น Brovey Transform, IHS Transform และ PCA วิธีการเหล่านี้ทำให้แน่ใจว่าภาพใหม่จะคงสีจาก Landsat-9 แต่ยังได้รับรายละเอียดเพิ่มเติมจากภาพแพนโครมาติกด้วย


เมื่อเราทำ Pan-sharpening เสร็จแล้ว เราก็จะได้ภาพใหม่ที่เป็นภาพสีที่มีรายละเอียดสูงขึ้น  โดยมีสีทั้งหมดจาก Landsat-9 และความคมชัดพิเศษจากภาพแบบแพนโครมาติก รูปภาพใหม่นี้สามารถนำไปใช้ได้หลายอย่าง เช่น ศึกษาพื้นดิน ค้นหาการเปลี่ยนแปลง หรือเพียงแค่ดูรายละเอียดเพิ่มเติมเกี่ยวกับโลก ดังนั้น การปรับความคมชัดของภาพเป็นวิธีหนึ่งในการทำให้ภาพจากดาวเทียมมีประโยชน์มากขึ้นสำหรับงานสำคัญทุกประเภท

In [ ]:
# ทำการ Authenticate และ initialize Earth Engine
import ee
import geemap
ee.Authenticate()
ee.Initialize(project='ee-chanita') #อย่าลืมเปลี่ยนชื่อโปรเจคของตัวเอง

In [ ]:
# กำหนดพื้นที่สนใจ
geometry = ee.Geometry.Point([98.95799098999555, 18.84423947416328])

# เรียกภาพ L9 ตัวอย่างแล้วดึง RGB และ Pan ออกมา
image = (ee.ImageCollection("LANDSAT/LC09/C02/T1_TOA")
         .filterDate('2022-01-01', '2022-03-30')
         .filterBounds(geometry)
         .sort('CLOUD_COVER')
         .first())

In [ ]:
# ทำการ Pan-Sharp
rgb = image.select('B4', 'B3', 'B2')
pan = image.select('B8')

# แปลงเป็น HSV, สลับในแถบ PAN และแปลงกลับเป็น RGB
huesat = rgb.rgbToHsv().select('hue', 'saturation')
upres = ee.Image.cat([huesat, pan]).hsvToRgb()

In [ ]:
# สร้างแผนที่
Map = geemap.Map(center=[18.84423947416328, 98.95799098999555], zoom=14)

# แสดง เลเยอร์ ก่อนและหลังโดยใช้พารามิเตอร์ vis เดียวกัน
Map.addLayer(rgb, {'max': 0.28}, 'Original')
Map.addLayer(pan, {'max': 0.28}, 'Pan')
Map.addLayer(upres, {'max': 0.28}, 'Pansharpened')
Map


Map(center=[18.84423947416328, 98.95799098999555], controls=(WidgetControl(options=['position', 'transparent_b…

คำถาม
ข้อเพื่อทดสอบความเข้าใจของนักศึกษาเกี่ยวกับเทคนิคการทำ Pan-sharpening ด้วยภาพ Landsat-9 จงตอบคำถามต่อไปนี้

1. อะไรคือเป้าหมายหลักของการปรับความคมชัดของภาพในการสำรวจระยะไกล โดยเฉพาะเมื่อใช้ภาพ Landsat-9 อธิบายว่าเหตุใดจึงมีความสำคัญในการประมวลผลภาพ

2. อธิบายขั้นตอนสำคัญที่เกี่ยวข้องกับการปรับความคมชัดด้วนเทคนิค Pan-sharpening ตั้งแต่การรับข้อมูลไปจนถึงการสร้างภาพที่ปรับความคมชัด เทคนิค Pan-sharpening มี กระบวนการสุ่มตัวอย่างใหม่ช่วยจัดแนวภาพหลายสเปกตรัมและภาพแพนโครมาติกอย่างไร


**1.อะไรคือเป้าหมายหลักของการปรับความคมชัดของภาพในการสำรวจระยะไกล โดยเฉพาะเมื่อใช้ภาพ Landsat-9 อธิบายว่าเหตุใดจึงมีความสำคัญในการประมวลผลภาพ**

เป้าหมายหลักของการปรับความคมชัดของภาพในการสำรวจระยะไกล คือ
*  เพื่อเพิ่มความแตกต่างของวัตถุในภาพ ช่วยให้พื้นที่ที่มีค่าการสะท้อนใกล้กัน ดูแตกต่างกันและชัดเจนมากขึ้น
*   เพื่อเน้นรายละเอียดเชิงพื้นที่ ทำให้ขอบเขตของถนน ลำน้ำ แนวชายฝั่ง หรือแปลงเกษตรเห็นชัดขึ้น โดยเฉพาะเมื่อใช้ภาพความละเอียดปานกลางอย่าง Landsat-9 (ความละเอียด 30 ม.)
*   เพื่อลดผลกระทบจากสภาพแวดล้อมและ sensor เช่น หมอก ควัน ความต่างของแสงเงา หรือสัญญาณรบกวนจากเซนเซอร์ เพื่อให้ภาพใกล้เคียงความจริงมากขึ้น
* เป็นการเตรียมภาพให้เหมาะกับการวิเคราะห์ขั้นต่อไป

การปรับความคมชัดของภาพมีความสำคัญในการประมวลผลภาพ เพราะช่วยให้ข้อมูลจากภาพดาวเทียม มีความชัดเจนและใกล้เคียงความเป็นจริงมากขึ้น ภาพดิบมักมีคอนทราสต์ต่ำหรือได้รับผลกระทบจากบรรยากาศและสัญญาณรบกวน ทำให้วัตถุหลายประเภทมีค่าการสะท้อนใกล้เคียงกันและแยกวัตถุได้ยาก เมื่อมีการปรับความคมชัด จะช่วยให้เห็นความแตกต่างของพื้นที่ได้ชัดขึ้น ทั้งสำหรับการมองด้วยสายตาและการวิเคราะห์ด้วยคอมพิวเตอร์ ส่งผลให้อัลกอริทึมจำแนกข้อมูลได้แม่นยำขึ้น ลดความคลาดเคลื่อนของข้อมูล













**2.อธิบายขั้นตอนสำคัญที่เกี่ยวข้องกับการปรับความคมชัดด้วยเทคนิค Pan-sharpening ตั้งแต่การรับข้อมูลไปจนถึงการสร้างภาพที่ปรับความคมชัด เทคนิค Pan-sharpening มี กระบวนการสุ่มตัวอย่างใหม่ช่วยจัดแนวภาพหลายสเปกตรัมและภาพแพนโครมาติกอย่างไร**

**ขั้นตอนการปรับความคมชัดด้วยเทคนิค Pan-sharpening มีดังนี้**

เริ่มจากการรับข้อมูล โดยเลือกภาพ Multispectral (MS) และ Panchromatic (PAN) จากดาวเทียมเดียวกัน เช่น Landsat-9 ซึ่ง MS มีความละเอียด 30 เมตร และ PAN มีความละเอียด 15 เมตร ต้องเป็นภาพพื้นที่เดียวกันและช่วงเวลาใกล้เคียงกันเพื่อลดความคลาดเคลื่อน

ต่อมาเป็นขั้นตอน Pre-processing ได้แก่ การปรับแก้เชิงรังสี (Radiometric correction) และการปรับแก้บรรยากาศ (Atmospheric correction) เพื่อให้ค่าการสะท้อนของภาพถูกต้องและเปรียบเทียบกันได้ หากไม่ทำขั้นตอนนี้ สีของภาพที่ได้หลังผสานอาจมีความผิดเพี้ยน

ทำการจัดแนวภาพ (Image Registration / Co-registration) เพื่อให้ตำแหน่งพิกเซลของภาพ MS และ PAN ตรงกันพอดีในเชิงพิกัดภูมิศาสตร์ หากภาพเหลื่อมกันแม้เล็กน้อย จะทำให้ภาพสุดท้ายเบลอหรือเกิดขอบซ้อน

ทำการปรับขนาดพิกเซลหรือการสุ่มตัวอย่างใหม่ (Resampling) โดยปรับภาพ MS จาก 30 เมตร ให้มีขนาดพิกเซลเท่ากับ PAN ที่ 15 เมตร เพื่อให้ทั้งสองภาพมีกริดพิกเซลเดียวกัน วิธีที่ใช้ เช่น Nearest Neighbor, Bilinear หรือ Cubic Convolution ขั้นตอนนี้ช่วยให้ข้อมูลจากทั้งสองภาพสามารถรวมกันได้อย่างแม่นยำในระดับพิกเซล

จากนั้นทำการผสานข้อมูล (Fusion Process) โดยใช้เทคนิค เช่น IHS, Brovey Transform และ PCA หลักการคือดึงรายละเอียดเชิงพื้นที่จาก PAN ไปเสริมในองค์ประกอบความสว่างของภาพ MS แล้วสร้างภาพใหม่ที่ยังคงข้อมูลสเปกตรัมเดิมมากที่สุด แต่มีความคมชัดเพิ่มขึ้น

และสุดท้ายเป็นการประเมินคุณภาพ (Quality Assessment) เพื่อตรวจสอบว่าภาพที่ได้มีรายละเอียดเพิ่มขึ้นจริง และสีหรือค่าการสะท้อนไม่บิดเบือนมากเกินไป อาจประเมินทั้งด้วยสายตาและค่าทางสถิติ

**กระบวนการสุ่มตัวอย่างใหม่ช่วยจัดแนวภาพโดย**
*   ปรับขนาดพิกเซลของภาพ MS ให้เท่ากับ PAN
*   คำนวณค่าพิกเซลใหม่ตามตำแหน่งพิกัดเดียวกัน
*   ทำให้พิกเซลของทั้งสองภาพซ้อนทับกันแบบ 1:1
